Connected to base (Python 3.12.7)

In [ ]:
from pathlib import Path

import pandas as pd

#from __future__ import annotations

from pathlib import Path

import pandas as pd
import numpy as np

from cei_abm.tru_sector_sector import (
    _safe_divide,
    build_tru_model_2,
    load_tru_data,
    transform_tru_to_sector_sector
)

In [ ]:
### Upload da matriz insumo-produto

data_dir = Path(r"D:\Pichau\OneDrive\Área de Trabalho\CEIS\nivel_20")

data = load_tru_data(data_dir=data_dir, year=2020, level=20)

transformed_data = transform_tru_to_sector_sector(data,validate=False)

In [ ]:
# Componentes de demanda:

household_consumption = transformed_data.household_consumption_sector + transformed_data.npo_consumption_sector
investment = transformed_data.gross_investment_sector + transformed_data.stocks_investment_sector
government_consumption = transformed_data.gov_cons_sector
exports = transformed_data.exports_sector

final_demand_sector = household_consumption + government_consumption + exports + investment

# Check: 
transformed_data.final_demand_sector / final_demand_sector

sum(final_demand_sector)

array([8815606.])

In [ ]:
output_market_prices = transformed_data.leontief_inverse @ final_demand_sector

output_market_prices

sum(output_market_prices)

array([15526868.])

In [ ]:
identity = np.eye(transformed_data.final_demand_sector.shape[0])

tax_coefficients = np.diagflat(_safe_divide(transformed_data.taxes_sector, output_market_prices))

trade_margin_coefficients = np.diagflat(
        _safe_divide(transformed_data.trade_margin_sector, output_market_prices)
)
    
transport_margin_coefficients = np.diagflat(
        _safe_divide(transformed_data.transport_margin_sector, output_market_prices)
)

basic_price_conversion = (
        identity
        - tax_coefficients
        - trade_margin_coefficients
        - transport_margin_coefficients
)

output_basic_price = basic_price_conversion @ output_market_prices

sum(output_basic_price)

array([14512208.])

In [ ]:
# Cálculo do market-share interno e externo por setor

ms_imp = transformed_data.imports_sector/output_basic_price

ms_dom = 1 - ms_imp

# Check:
#ms_dom * output_basic_price/data.production.sum(axis=0).reshape(-1,1)

In [ ]:
sector_output = output_basic_price * ms_dom

In [ ]:
#data.value_added_components

value_added_table = pd.DataFrame(
    data.value_added_components,
    index=pd.Index(data.va_components_names, name="componente"),
    columns=pd.Index(data.sector_names, name="setor"),
)

print(value_added_table)

setor                                               A\nAgricultura, pecuária, produção florestal, pesca e aquicultura  \
componente                                                                                                              
Valor adicionado bruto ( PIB )                                                               434621.0                   
Remunerações                                                                                  56426.0                   
Salários                                                                                      48460.0                   
Contribuições sociais efetivas                                                                 7966.0                   
Previdência oficial /FGTS                                                                      7966.0                   
Previdência privada                                                                               0.0                   
Contribuições sociais imputadas 

In [ ]:
# Cálculo das razões:

value_added_ratios = value_added_table.div(value_added_table.loc["Valor da produção"], axis=1)

In [ ]:
# Produção setorial com os nomes dos setores
sector_output_series = pd.Series(
    np.asarray(sector_output).reshape(-1),
    index=data.sector_names,
    name="produção",
)

# Atualização dos componentes por setor
value_added = value_added_ratios.mul(
    sector_output_series,
    axis="columns",
)

value_added

setor,"A\nAgricultura, pecuária, produção florestal, pesca e aquicultura",B\nIndústrias extrativas,C\nIndústrias de transformação,D\nEletricidade e gás,"E\nÁgua, esgoto, atividades de gestão de resíduos e descontaminação",F\nConstrução,G\nComércio; reparação de veículos automotores e motocicletas,"H\nTransporte, armazenagem e correio",I\nAlojamento e alimentação,J\nInformação e comunicação,"K\nAtividades financeiras, de seguros e serviços relacionados",L\nAtividades imobiliárias,"M\nAtividades científicas, profissionais e técnicas",N\nAtividades administrativas e serviços complementares,"O\nAdministração pública, defesa e seguridade social",P\nEducação,Q\nSaúde humana e serviços sociais,"R\nArtes, cultura, esporte e recreação",S\nOutras atividades de serviços,T\nServiços domésticos
componente,,,,,,,,,,,,,,,,,,,,
Valor adicionado bruto ( PIB ),434621.0,193615.0,813689.0,150795.0,58317.0,267921.0,825346.0,273239.0,117465.0,237574.0,454550.0,656013.0,258249.0,265586.0,668908.0,421906.0,331297.0,21512.0,84860.0,59474.0
Remunerações,56426.0,31030.0,431907.0,18666.0,23674.0,110214.0,404268.0,144416.0,59275.0,106305.0,181285.0,9024.0,109900.0,170530.0,567850.0,402499.0,249411.0,11879.0,44310.0,59474.0
Salários,48460.0,23545.0,342665.0,13453.0,17927.0,90262.0,322109.0,115820.0,49789.0,84430.0,140274.0,7050.0,89619.0,136337.0,408467.0,326544.0,208790.0,10266.0,38266.0,57888.0
Contribuições sociais efetivas,7966.0,7485.0,89242.0,5213.0,5747.0,19952.0,82159.0,28596.0,9486.0,21875.0,41011.0,1974.0,20281.0,34193.0,76174.0,68619.0,33401.0,1613.0,6044.0,1586.0
Previdência oficial /FGTS,7966.0,6442.0,85542.0,3981.0,5222.0,19642.0,81427.0,27010.0,9425.0,20152.0,34959.0,1927.0,19052.0,33917.0,74045.0,68254.0,33298.0,1577.0,5957.0,1586.0
Previdência privada,0.0,1043.0,3700.0,1232.0,525.0,310.0,732.0,1586.0,61.0,1723.0,6052.0,47.0,1229.0,276.0,2129.0,365.0,103.0,36.0,87.0,0.0
Contribuições sociais imputadas,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,83209.0,7336.0,7220.0,0.0,0.0,0.0
Excedente operacional bruto e rendimento misto bruto,380742.0,159960.0,353922.0,128931.0,33709.0,154409.0,408143.0,123266.0,56553.0,125590.0,264003.0,646363.0,144947.0,90121.0,100780.0,17168.0,78772.0,9364.0,39460.0,0.0
Rendimento misto bruto,203321.0,246.0,26998.0,0.0,1342.0,65864.0,82745.0,29289.0,42901.0,11697.0,3241.0,4446.0,54428.0,9330.0,0.0,7282.0,54843.0,6843.0,28548.0,0.0


In [ ]:
# Componentes que entram na CEI abaixo:

# Impostos indiretos que entram na TRU

# Impostos indiretos não-financeiras e financeiras

# Cálculo da taxa de impostos setorial 

original_output_market_prices = transformed_data.leontief_inverse @ transformed_data.final_demand_sector 
tax_rate = transformed_data.taxes_sector/original_output_market_prices

In [ ]:
taxes = tax_rate * output_market_prices
taxes

sum(taxes)

array([1014660.])

In [ ]:
ffirms_taxes = taxes[10, 0] # Valor fica diferente da TRU, pq a TRU considera produtos x setor. Aqui está setor x setor, gerando diferença marginal.
ffirms_taxes

print(f"Imposto de empresas financeiras: {ffirms_taxes:.2f}")

nffirms_taxes = taxes.sum() - ffirms_taxes

print(f"Imposto de empresas não financeiras: {nffirms_taxes:.2f}")

Imposto de empresas financeiras: 68029.85
Imposto de empresas não financeiras: 946630.15


In [ ]:
# Cálculo do VA a preços de mercado por setor institucional

va_ff = value_added.iloc[0,10] + ffirms_taxes

va_nf = value_added.iloc[0,:].sum() - value_added.iloc[0,10] + nffirms_taxes

# Check:
va_ff + va_nf

7609597.0

In [ ]:
### Upload da CEI

ARQUIVO_ENTRADA = Path(
    r"D:\Pichau\OneDrive\Área de Trabalho\CEIS\CEI2020_adaptado.xlsx"
)
NOME_ABA = "Python"


# Cria o DataFrame dados a partir da planilha.
dados = pd.read_excel(
    ARQUIVO_ENTRADA,
    sheet_name=NOME_ABA
)

# Mantém linhas com algum valor diferente de zero,
# desconsiderando a primeira coluna.
condicao = (
    dados.iloc[:, 1:]
    .fillna(0)
    .ne(0)
    .any(axis=1)
)

# Seleciona as 11 primeiras colunas e refaz o índice.
fluxos_original = (
    dados.loc[condicao, dados.columns[:11]]
    .copy()
    .reset_index(drop=True)
)

# Cria uma cópia independente.
fluxos_adaptaveis = fluxos_original.copy()

# Faz a alteração somente na tabela adaptável.
fluxos_adaptaveis.iloc[16, 1] = (
    fluxos_adaptaveis.iloc[1:15, 1].sum()
    - fluxos_adaptaveis.iloc[1:15, 2].sum()
)

print(fluxos_original)
print(fluxos_adaptaveis)

                                           Unnamed: 0 Household Household.1  \
0                                                 NaN   Entrada       Saída   
1    Valor adicionado bruto/Produto interno bruto (1)         0           0   
2                                            Salários   2531961           0   
3                      Contribuições sociais efetivas    562617           0   
4     Impostos, líquidos de subsídios, sobre produtos         0           0   
5     Outros impostos, líquidos de subsídios, sobr...         0           0   
6                                               Juros    249546      205143   
7                    Rendas distribuídas das empresas   1888670           0   
8   Impostos correntes sobre a renda, patrimônio, ...         0      362776   
9                               Contribuições sociais         0      895266   
10    Benefícios de seguridade social em numerário...   1254637           0   
11    Outros benefícios de seguro social // Previd..

In [ ]:
outros_impostos_ff = value_added.iloc[10,10] + value_added.iloc[11,10]
outros_impostos_nf = value_added.iloc[10,:].sum() + value_added.iloc[11,:].sum()  - outros_impostos_ff

print(outros_impostos_nf)

outros_impostos_ratio_ff = outros_impostos_ff/va_ff
outros_impostos_ratio_nf = outros_impostos_nf/va_nf

77129.0


In [ ]:
outros_impostos_ff = value_added.iloc[10,10] + value_added.iloc[11,10]
outros_impostos_nf = value_added.iloc[10,:].sum() + value_added.iloc[11,:].sum()  - outros_impostos_ff

print(outros_impostos_ff)

outros_impostos_ratio_ff = outros_impostos_ff/va_ff
outros_impostos_ratio_nf = outros_impostos_nf/va_nf

9261.999999999998


In [ ]:
outros_impostos_ff = value_added.iloc[10,10] + value_added.iloc[11,10]
outros_impostos_nf = value_added.iloc[10,:].sum() + value_added.iloc[11,:].sum()  - outros_impostos_ff

print(outros_impostos_nf)

outros_impostos_ratio_ff = outros_impostos_ff/va_ff
outros_impostos_ratio_nf = outros_impostos_nf/va_nf

77129.0


In [ ]:
# Linha 1 - Valor adicionado bruto: preços de mercado

value_added

setor,"A\nAgricultura, pecuária, produção florestal, pesca e aquicultura",B\nIndústrias extrativas,C\nIndústrias de transformação,D\nEletricidade e gás,"E\nÁgua, esgoto, atividades de gestão de resíduos e descontaminação",F\nConstrução,G\nComércio; reparação de veículos automotores e motocicletas,"H\nTransporte, armazenagem e correio",I\nAlojamento e alimentação,J\nInformação e comunicação,"K\nAtividades financeiras, de seguros e serviços relacionados",L\nAtividades imobiliárias,"M\nAtividades científicas, profissionais e técnicas",N\nAtividades administrativas e serviços complementares,"O\nAdministração pública, defesa e seguridade social",P\nEducação,Q\nSaúde humana e serviços sociais,"R\nArtes, cultura, esporte e recreação",S\nOutras atividades de serviços,T\nServiços domésticos
componente,,,,,,,,,,,,,,,,,,,,
Valor adicionado bruto ( PIB ),434621.0,193615.0,813689.0,150795.0,58317.0,267921.0,825346.0,273239.0,117465.0,237574.0,454550.0,656013.0,258249.0,265586.0,668908.0,421906.0,331297.0,21512.0,84860.0,59474.0
Remunerações,56426.0,31030.0,431907.0,18666.0,23674.0,110214.0,404268.0,144416.0,59275.0,106305.0,181285.0,9024.0,109900.0,170530.0,567850.0,402499.0,249411.0,11879.0,44310.0,59474.0
Salários,48460.0,23545.0,342665.0,13453.0,17927.0,90262.0,322109.0,115820.0,49789.0,84430.0,140274.0,7050.0,89619.0,136337.0,408467.0,326544.0,208790.0,10266.0,38266.0,57888.0
Contribuições sociais efetivas,7966.0,7485.0,89242.0,5213.0,5747.0,19952.0,82159.0,28596.0,9486.0,21875.0,41011.0,1974.0,20281.0,34193.0,76174.0,68619.0,33401.0,1613.0,6044.0,1586.0
Previdência oficial /FGTS,7966.0,6442.0,85542.0,3981.0,5222.0,19642.0,81427.0,27010.0,9425.0,20152.0,34959.0,1927.0,19052.0,33917.0,74045.0,68254.0,33298.0,1577.0,5957.0,1586.0
Previdência privada,0.0,1043.0,3700.0,1232.0,525.0,310.0,732.0,1586.0,61.0,1723.0,6052.0,47.0,1229.0,276.0,2129.0,365.0,103.0,36.0,87.0,0.0
Contribuições sociais imputadas,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,83209.0,7336.0,7220.0,0.0,0.0,0.0
Excedente operacional bruto e rendimento misto bruto,380742.0,159960.0,353922.0,128931.0,33709.0,154409.0,408143.0,123266.0,56553.0,125590.0,264003.0,646363.0,144947.0,90121.0,100780.0,17168.0,78772.0,9364.0,39460.0,0.0
Rendimento misto bruto,203321.0,246.0,26998.0,0.0,1342.0,65864.0,82745.0,29289.0,42901.0,11697.0,3241.0,4446.0,54428.0,9330.0,0.0,7282.0,54843.0,6843.0,28548.0,0.0


In [ ]:
# Linha 1 - Valor adicionado bruto: preços de mercado

value_added[1,10]

KeyError: (1, 10)

In [ ]:
# Linha 1 - Valor adicionado bruto: preços de mercado

value_added.iloc[1,10]

181284.99999999994

In [ ]:
# Linha 1 - Valor adicionado bruto: preços de mercado

value_added.iloc[0,10]

454549.99999999994

In [ ]:
fluxos_adaptaveis

,Unnamed: 0,Household,Household.1,Gov,Gov.1,Ffirms,Ffirms.1,Firms,Firms.1,ExtSector,ExtSector.1
0,NaN,Entrada,Saída,Entrada,Saída,Entrada,Saída,Entrada,Saída,Entrada,Saída
1,Valor adicionado bruto/Produto interno bruto (1),0,0,0,0,454550,0,7155047,0,1206009,1252049
2,Salários,2531961,0,0,0,0,140274,0,2391687,0,0
3,Contribuições sociais efetivas,562617,0,0,0,0,41011,0,521606,0,0
4,"Impostos, líquidos de subsídios, sobre produtos",0,0,1014660,0,0,67488.0,0,947172,0,0
5,"Outros impostos, líquidos de subsídios, sobr...",0,0,86391,0,0,9262,0,77129,0,0
6,Juros,249546,205143,191053,446981,1235025,1060460,246213,282899,106219,32573
7,Rendas distribuídas das empresas,1888670,0,0,0,0,153250,0,1821126,85706,0
8,"Impostos correntes sobre a renda, patrimônio, ...",0,362776,917257,0,0,42715,0,511766,0,0
9,Contribuições sociais,0,895266,800730,0,94536,0,0,0,0,0


In [ ]:
# Linha 4 - Impostos, líquidos de subsídios, sobre produtos

taxes[10]

array([68029.85420352])

In [ ]:
# Linha 4 - Impostos, líquidos de subsídios, sobre produtos

fluxos_adaptaveis[4,6] = taxes[10]
fluxos_adaptaveis[4,8] = taxes.sum() - taxes[10]

ValueError: Length of values (1) does not match length of index (17)

In [ ]:
taxes

array([[ 20273.83725876],
       [  7324.22903446],
       [614927.71757278],
       [ 67120.03534659],
       [  3843.01515114],
       [ 29621.77094982],
       [ 18876.61491844],
       [ 30114.1552275 ],
       [ 28349.35215123],
       [ 56063.6646947 ],
       [ 68029.85420352],
       [  1131.8547489 ],
       [ 21325.39204748],
       [ 25127.31213585],
       [  2924.74999656],
       [  3254.47322962],
       [  7385.92704424],
       [  7023.66814289],
       [  1942.37614551],
       [     0.        ]])

In [ ]:
taxes.sum()

1014660.0

In [ ]:
# Linha 4 - Impostos, líquidos de subsídios, sobre produtos

fluxos_adaptaveis[4,6] = taxes[1,10]
fluxos_adaptaveis[4,8] = taxes.sum() - taxes[1,10]

IndexError: index 10 is out of bounds for axis 1 with size 1